In [2]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
REPO_DIR = Path('/content/Yolo-ST-GCN')
BRANCH = 'refactor-1'

# Install lightweight dependencies needed for smoke checks.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'numpy', 'scipy', 'torch', '-q'],
    check=True,
)

# Clone or update exact branch used for current development.
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', os.getcwd())
print('Branch:', BRANCH)

Repo ready: /content/Yolo-ST-GCN
Branch: refactor-1


In [3]:
# !git pull

In [4]:
!python scripts/build_gym99_from_gym288.py --experiment_config configs/experiments/gym99_build_from_gym288.json

usage: build_gym99_from_gym288.py [-h] [--experiment_config EXPERIMENT_CONFIG]
                                  --gym288_dataset_path GYM288_DATASET_PATH
                                  --gym99_dataset_path GYM99_DATASET_PATH
                                  [--gym288_categories_url GYM288_CATEGORIES_URL]
                                  [--gym99_categories_url GYM99_CATEGORIES_URL]
                                  [--disable_neighbor_fallback]
build_gym99_from_gym288.py: error: the following arguments are required: --gym288_dataset_path, --gym99_dataset_path


In [7]:
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from huggingface_hub import snapshot_download

download_dir = Path('/content/Gym288-skeleton')
if not (download_dir.exists() and any(download_dir.iterdir())):
    print('Downloading Gym288-skeleton...')
    download_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(download_dir),
        local_dir_use_symlinks=False,
    )
else:
    print('Gym288-skeleton already exists.')

pkl_candidates = sorted(download_dir.rglob('*.pkl'))
if not pkl_candidates:
    raise FileNotFoundError('No .pkl file found for Gym288 dataset.')

gym288_pkl = str(pkl_candidates[0])
print('Gym288 pickle:', gym288_pkl)

Gym288-skeleton already exists.
Gym288 pickle: /content/Gym288-skeleton/gym_288_skeleton.pkl


In [9]:
!python scripts/train_gym99.py \
--auto_build_from_gym288 \
--gym288_dataset_path /content/Gym288-skeleton/gym_288_skeleton.pkl \
--dataset_path /content/Gym99-from-Gym288/gym99_from_gym288.pkl \
--out_dir outputs/refactor1_2ep/gym99_coco18_2s \
--epochs 2 \
--batch_size 256 \
--lr 0.001 \
--num_workers 0 \
--joint_spec_name coco18 \
--use_two_stream

Building Gym99-from-Gym288 pickle...
Gym99 mapping stats: direct=34240 minus1=1625 plus1=800 train=27624 test=9041
Device: cuda
Loading Gym99-skeleton dataset...
Loaded 36665 samples  train=27624  test=9041
num_classes=99 (inferred=99)
DataLoader num_workers=0  two_stream=True
Epoch 1/2  train_loss=3.9843  train_acc=0.0819  val_loss=3.3708  val_acc=0.1435  val_f1=0.0324
Epoch 2/2  train_loss=3.0897  train_acc=0.2124  val_loss=2.5769  val_acc=0.2980  val_f1=0.1267
Saved weights: outputs/refactor1_2ep/gym99_coco18_2s/stgcn_gym99_coco18_2s.pth
Test Top-1 Accuracy: 0.2980
Test Macro-F1     : 0.1267


In [13]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 618 bytes | 618.00 KiB/s, done.
From https://github.com/schizocatto/Yolo-ST-GCN
   835a80a..79d93f2  refactor-1 -> origin/refactor-1
Updating 835a80a..79d93f2
Fast-forward
 src/train.py | 15 +++++++++++++--
 1 file changed, 13 insertions(+), 2 deletions(-)


In [17]:
!python scripts/train_gym99.py \
--auto_build_from_gym288 \
--gym288_dataset_path /content/Gym288-skeleton/gym_288_skeleton.pkl \
--dataset_path /content/Gym99-from-Gym288/gym99_from_gym288.pkl \
--out_dir outputs/refactor1_2ep/gym99_coco18_2s \
--epochs 30 \
--batch_size 256 \
--lr 0.001 \
--num_workers 0 \
--max_train_samples 8192 \
--max_val_samples 2048 \
--joint_spec_name coco18 \
# --use_two_stream \
# --train_data_mode preload_vram

Building Gym99-from-Gym288 pickle...
Gym99 mapping stats: direct=34240 minus1=1625 plus1=800 train=27624 test=9041
Device: cuda
Loading Gym99-skeleton dataset...
Loaded 36665 samples  train=8192  test=2048
num_classes=99 (inferred=99)
DataLoader num_workers=0  two_stream=False  train_data_mode=standard
Epoch 1/30  train_loss=4.4859  train_acc=0.0542  val_loss=4.0488  val_acc=0.0723  val_f1=0.0051
Epoch 2/30  train_loss=3.9057  train_acc=0.0814  val_loss=3.6632  val_acc=0.1313  val_f1=0.0099
Epoch 3/30  train_loss=3.4569  train_acc=0.1426  val_loss=3.0657  val_acc=0.2085  val_f1=0.0359
Epoch 4/30  train_loss=2.9902  train_acc=0.2133  val_loss=2.6984  val_acc=0.2720  val_f1=0.0710
Epoch 5/30  train_loss=2.4508  train_acc=0.3234  val_loss=2.4880  val_acc=0.3325  val_f1=0.1066
Epoch 6/30  train_loss=2.1148  train_acc=0.3964  val_loss=2.0410  val_acc=0.4043  val_f1=0.1500
Epoch 7/30  train_loss=1.8081  train_acc=0.4520  val_loss=1.9058  val_acc=0.4365  val_f1=0.1739
Epoch 8/30  train_loss=1